# 03 - Write shards for benchmark datasets (OP3, Tahoe, Novartis)

Mirrors `lpm_style/notebooks_work/03_LPM_style_write_shards.ipynb`, but limited to the three benchmark datasets we added in notebooks 01 and 02. The existing LPM plibdata folders (l1000_*, cigs_*, vcpi_*, gdpx2_*, sciplex_*, dili_train_*) are **not touched**.

**Inputs:**
- `lpm_style/.plib_cache/annotations/df_annot_split_extended.parquet` (from notebook 02)
- `lpm_style/.plib_cache/raw_datasets/{op3, tahoe, novartis}/*.parquet` (from notebook 01)

**Outputs (extend existing plibdata, alongside `l1000_*`, `cigs_*`, ...):**
- `lpm_style/.plib_cache/plibdata/op3_<context>/`
- `lpm_style/.plib_cache/plibdata/tahoe_<context>/`
- `lpm_style/.plib_cache/plibdata/novartis_<context>/`

In [1]:
import json
import shutil
from collections import defaultdict
from pathlib import Path

import numpy as np
import os
import pandas as pd
from tqdm import tqdm

In [2]:
SHARDSIZE = 200_000

PLIBDATA_ROOT = '/home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets'
SHARDS_ROOT   = Path('/home/icb/olga.novitskaia/lpm_style/.plib_cache/plibdata')
ANNOT         = '/home/icb/olga.novitskaia/lpm_style/.plib_cache/annotations/df_annot_split_extended.parquet'

BENCHMARK_KEYS = ['op3', 'tahoe', 'novartis']

JOIN_KEYS  = ['dataset', 'context', 'perturbation']
SHARD_COLS = ['dataset', 'context', 'perturbation', 'readout', 'log_dose', 'time', 'value', 'split']

In [3]:
df_annot_split = pd.read_parquet(ANNOT)
df_annot_split['perturbation'] = df_annot_split['perturbation'].astype(str)
df_annot_split['context']      = df_annot_split['context'].astype(str)
print(df_annot_split.groupby(['dataset', 'split']).size().unstack(fill_value=0))

split                                 test  train    val
dataset                                                 
CIGS MCE                              3281  15274   3264
CIGS TCM                               559   2527    523
Ginkgo GDPx2                            54    241     49
Ginkgo VCPI vcpi-0001 (tvc-bhr-009)      0   2269      3
Ginkgo VCPI vcpi-0002 (tvc-kdl-010)      1   1487      0
LINCS_phase1_level3_epsilon          17243  80858  17410
LINCS_phase2_level3                   2383  10914   2294
Novartis MoABox DRUG-seq              1788   8024   1648
dilimap_train                           35    175     40
op3                                     66    291     58
srivatsan20_sciplex3                    90    392     78
tahoe100                              5672  28220   6444


## Helpers (copied verbatim from `lpm_style/notebooks_work/03`)

In [4]:
def _append_context_chunk(
    ctx_df: pd.DataFrame,
    out_dir: Path,
    shardsize: int,
    next_shard_id: int,
):
    """Append one chunk of rows to a (dataset, context) folder as plib-style shards.

    The folder is NOT wiped here — call `_reset_context_folder(out_dir)` once before the first
    chunk if you want a clean slate. Within a chunk, every emitted shard has a single split
    and at most `shardsize` rows. Shards from different chunks may belong to the same split,
    so a folder can contain several smaller-than-`shardsize` 'tail' shards.

    Returns (next_shard_id_after_chunk, metadata_rows_for_chunk).
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    long = ctx_df[SHARD_COLS].copy()
    long['perturbation'] = long['perturbation'].astype(str)
    long = long.sort_values('split', kind='mergesort').reset_index(drop=True)
    long['shard_id_per_split'] = long.groupby('split').cumcount() // shardsize
    combo = long['split'].astype(str) + '::' + long['shard_id_per_split'].astype(str)
    long['shard_id'] = pd.factorize(combo)[0] + next_shard_id
    long = long.drop(columns='shard_id_per_split')

    metadata_rows = []
    max_id = next_shard_id - 1
    for shard_id, shard_df in long.groupby('shard_id', sort=True):
        shard_name = f'shard_{int(shard_id):06d}.parquet'
        shard_df.drop(columns=['split', 'shard_id']).reset_index(drop=True).to_parquet(
            out_dir / shard_name, engine='pyarrow', index=False,
        )
        metadata_rows.append({
            'shard_path':    f'{out_dir.name}/{shard_name}',
            'size':          int(len(shard_df)),
            'split':         str(shard_df['split'].iloc[0]),
            'context':       str(shard_df['context'].iloc[0]),
            'datasets':      str(shard_df['dataset'].iloc[0]),
            'perturbations': sorted({p for s in shard_df['perturbation'] for p in s.split('+')}),
            'log_doses':     sorted(shard_df['log_dose'].dropna().unique().tolist()),
            'times':         sorted(shard_df['time'].dropna().unique().tolist()),
            'readouts':      shard_df['readout'].unique().tolist(),
        })
        max_id = max(max_id, int(shard_id))

    return max_id + 1, metadata_rows


def _reset_context_folder(out_dir: Path, shardsize: int) -> None:
    """Wipe a (dataset, context) folder and write info.json immediately so the folder is
    self-describing even before the first metadata.parquet is finalized."""
    out_dir.mkdir(parents=True, exist_ok=True)
    for f in out_dir.iterdir():
        (shutil.rmtree if f.is_dir() else Path.unlink)(f)
    (out_dir / 'info.json').write_text(json.dumps({
        'PDATA_FORMAT_VERSION': 1,
        'CONTEXT_MODULE_HASH': '',
        'SHARDSIZE': shardsize,
    }))


def _finalize_context_folder(out_dir: Path, metadata_rows: list) -> None:
    """Write metadata.parquet for a folder once all chunks have been appended."""
    pd.DataFrame(metadata_rows).to_parquet(out_dir / 'metadata.parquet', engine='pyarrow', index=False)

## Drive sharding for the three benchmark folders only

Only `BENCHMARK_KEYS` source folders are processed; the existing LPM `plibdata/` shards (`l1000_*`, `cigs_*`, `vcpi_*`, `gdpx2_*`, `sciplex_*`, `dili_train_*`) are untouched. A new run wipes and rewrites only the three new datasets' folders, so it is idempotent.

In [5]:
def _folder_name(dataset_folder: str, context: str) -> str:
    return f'{dataset_folder}_{context}'

SHARDS_ROOT.mkdir(parents=True, exist_ok=True)
print(f'shards will be written under: {SHARDS_ROOT}')
print(f'processing benchmark folders: {BENCHMARK_KEYS}')

shards will be written under: /home/icb/olga.novitskaia/lpm_style/.plib_cache/plibdata
processing benchmark folders: ['op3', 'tahoe', 'novartis']


In [6]:
split_lookup = df_annot_split[JOIN_KEYS + ['split']].drop_duplicates(JOIN_KEYS)

shard_counters       = defaultdict(int)
metadata_per_folder  = defaultdict(list)
folder_initialized   = set()

for dataset_folder in tqdm(BENCHMARK_KEYS, desc='datasets'):
    dataset_dir = Path(PLIBDATA_ROOT) / dataset_folder
    if not dataset_dir.is_dir():
        print(f'  [skip] {dataset_folder}: not found at {dataset_dir}')
        continue
    files = sorted(f for f in os.listdir(dataset_dir) if f.endswith('.parquet'))
    if not files:
        print(f'  [skip] {dataset_folder}: no parquet files')
        continue

    touched_in_this_dataset = set()

    for file in tqdm(files, desc=dataset_folder, leave=False):
        df = pd.read_parquet(dataset_dir / file)
        if df.empty:
            continue
        df['perturbation'] = df['perturbation'].astype(str)
        df['context']      = df['context'].astype(str)

        n_before = len(df)
        df = df.merge(split_lookup, on=JOIN_KEYS, how='inner')
        n_lost = n_before - len(df)
        if n_lost:
            print(f'  [warn] {dataset_folder}/{file}: {n_lost} rows had no split assignment, dropped')
        if df.empty:
            continue

        for context, ctx_df in df.groupby('context', sort=True):
            folder_name = _folder_name(dataset_folder, str(context))
            out_dir = SHARDS_ROOT / folder_name

            if folder_name not in folder_initialized:
                _reset_context_folder(out_dir, SHARDSIZE)
                folder_initialized.add(folder_name)

            next_id, meta = _append_context_chunk(
                ctx_df, out_dir, SHARDSIZE, shard_counters[folder_name],
            )
            shard_counters[folder_name] = next_id
            metadata_per_folder[folder_name].extend(meta)
            touched_in_this_dataset.add(folder_name)

    for folder_name in touched_in_this_dataset:
        _finalize_context_folder(SHARDS_ROOT / folder_name, metadata_per_folder[folder_name])

total_shards = sum(len(v) for v in metadata_per_folder.values())
total_folders = len(metadata_per_folder)
print(f'done: wrote {total_shards} shards across {total_folders} dataset_context folders -> {SHARDS_ROOT}')

datasets: 100%|██████████| 3/3 [1:02:38<00:00, 1252.71s/it]

done: wrote 5813 shards across 53 dataset_context folders -> /home/icb/olga.novitskaia/lpm_style/.plib_cache/plibdata


## Verify the new folders

In [7]:
for item in sorted(os.listdir(SHARDS_ROOT)):
    if any(item.startswith(k + '_') for k in BENCHMARK_KEYS):
        print(f'      - {item}')

      - novartis_CVCL_0042
      - op3_B cells
      - op3_Myeloid cells
      - op3_NK cells
      - op3_T cells
      - tahoe_CVCL_0023
      - tahoe_CVCL_0028
      - tahoe_CVCL_0069
      - tahoe_CVCL_0099
      - tahoe_CVCL_0131
      - tahoe_CVCL_0152
      - tahoe_CVCL_0179
      - tahoe_CVCL_0218
      - tahoe_CVCL_0292
      - tahoe_CVCL_0293
      - tahoe_CVCL_0320
      - tahoe_CVCL_0332
      - tahoe_CVCL_0334
      - tahoe_CVCL_0359
      - tahoe_CVCL_0366
      - tahoe_CVCL_0371
      - tahoe_CVCL_0397
      - tahoe_CVCL_0399
      - tahoe_CVCL_0428
      - tahoe_CVCL_0459
      - tahoe_CVCL_0480
      - tahoe_CVCL_0504
      - tahoe_CVCL_0546
      - tahoe_CVCL_1055
      - tahoe_CVCL_1056
      - tahoe_CVCL_1094
      - tahoe_CVCL_1097
      - tahoe_CVCL_1098
      - tahoe_CVCL_1119
      - tahoe_CVCL_1125
      - tahoe_CVCL_1239
      - tahoe_CVCL_1285
      - tahoe_CVCL_1381
      - tahoe_CVCL_1478
      - tahoe_CVCL_1495
      - tahoe_CVCL_1517
      - tahoe_CVCL_154